This part of the pipeline estimates the genomic divergence rate of each rRNA lineage using Panstripe.

### Paths and parameters

#### Pipeline input folders

In [ ]:
pa.file = "03-pangenomes/all/gene_presence_absence.Rtab"
grouping.file = "02-GTDB/filtered_classification_table"
subtrees.folder = "04-core-phylogeny/subtrees"

#### Pipeline output folders

In [ ]:
task_root = "08-temporal-analysis"
system(paste0('mkdir -p ', task_root), intern = TRUE)

#### Tool pointers and parameters

In [ ]:
set.seed(127)

In [ ]:
library(panstripe)
library(ape)
library(ggplot2)

### Load files and metadata

#### Presence/absence files

In [ ]:
pa = read_rtab(pa.file)

In [ ]:
nrow(pa)

#### Grouping

In [ ]:
grouping = read.table(grouping.file, col.names = c('accession', 'group'))

In [ ]:
grouping

#### Phylogenies

In [ ]:
tree.files = sapply(c(unique(grouping$group), 'all'), function(x) paste('04-core-phylogeny', 'subtrees', paste0(x, '.contree'), sep = "/"))

In [ ]:
tree.files

In [ ]:
trees = lapply(tree.files, read.tree)

### Fitting genomic divergence models

using Gaussian GLMs for robustness and ease of convergence

In [ ]:
extract_subpa = function(pa, requested_group) {
    subpa = pa[grouping[grouping$group == requested_group,]$accession,]
    return(subpa)
}

In [ ]:
fit.clostridiales = panstripe(extract_subpa(pa, 'Clostridiales'), trees[['Clostridiales']], family = "gaussian")
fit.lachnospirales = panstripe(extract_subpa(pa, 'Lachnospirales'), trees[['Lachnospirales']], family = "gaussian")
fit.oscillospirales = panstripe(extract_subpa(pa, 'Oscillospirales'), trees[['Oscillospirales']], family = "gaussian")
fit.peptostreptococcales = panstripe(extract_subpa(pa, 'Peptostreptococcales'), trees[['Peptostreptococcales']], family = "gaussian")

In [ ]:
plot_residuals(fit.clostridiales)

In [ ]:
plot_residuals(fit.lachnospirales)

In [ ]:
plot_residuals(fit.oscillospirales)

In [ ]:
plot_residuals(fit.peptostreptococcales)

In [ ]:
fit.clostridiales$summary

In [ ]:
fit.lachnospirales$summary

In [ ]:
fit.oscillospirales$summary

In [ ]:
fit.peptostreptococcales$summary

In [ ]:
svg(paste(task_root, 'panstripe_cumulative_pangenome.svg', sep = "/"))
plot_pangenome_cumulative(list(Clostridiales = fit.clostridiales, 
                               Lachnospirales = fit.lachnospirales, 
                               Oscillospirales = fit.oscillospirales, 
                               Peptostreptococcales = fit.peptostreptococcales))
dev.off()

#### Statistically comparing the model fits

In [ ]:
compare_pangenomes(fit.clostridiales, fit.lachnospirales, family = "gaussian")

In [ ]:
compare_pangenomes(fit.clostridiales, fit.oscillospirales, family = "gaussian")

In [ ]:
compare_pangenomes(fit.clostridiales, fit.peptostreptococcales, family = "gaussian")

In [ ]:
compare_pangenomes(fit.lachnospirales, fit.oscillospirales, family = "gaussian")

In [ ]:
compare_pangenomes(fit.lachnospirales, fit.peptostreptococcales, family = "gaussian")

In [ ]:
compare_pangenomes(fit.oscillospirales, fit.peptostreptococcales, family = "gaussian")

#### Saving fits

In [ ]:
save.image(file = paste(task_root, "environment.RData", sep = "/"))

In [2]:
sessionInfo()

R version 4.3.2 (2023-10-31)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 22.04.4 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.20.so;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Brussels
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] ggplot2_3.5.0   ape_5.7-1       panstripe_0.2.0

loaded via a namespace (and not attached):
 [1] crayon_1.5.2     vctrs_0.6.5      nlme_3.1-164     cli_3.6.2 